In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, classification_report
from sentence_transformers import SentenceTransformer
import warnings
warnings.filterwarnings('ignore')

In [ ]:
# ==========================================
# 1. TẢI DỮ LIỆU
# ==========================================
print("Đang tải dữ liệu...")
train_df = pd.read_csv('../data/train.csv')
test_df = pd.read_csv('../data/test.csv')

# Điền rỗng cho các giá trị NaN
text_cols = ['title', 'abstract', 'authors', 'venue']
for col in text_cols:
    train_df[col] = train_df[col].fillna('')
    test_df[col] = test_df[col].fillna('')

Đang tải dữ liệu...


In [ ]:
# ==========================================
# 2. GHÉP CHUỖI ĐỂ ĐƯA VÀO BỘ ĐỌC HIỂU SBERT
# ==========================================
# SBERT làm việc cực tốt với các đoạn văn bản dài có ngữ nghĩa.
# Ta sẽ cấu trúc nó thành dạng "Title: ... Abstract: ..." để AI dễ đọc.
def create_text_for_bert(row):
    parts = []

    # Chỉ thêm vào nếu cột có nội dung thực sự (sau khi loại bỏ khoảng trắng)
    if str(row['title']).strip():
        parts.append(f"Title: {str(row['title']).strip()}")

    if str(row['abstract']).strip():
        parts.append(f"Abstract: {str(row['abstract']).strip()}")

    if str(row['authors']).strip():
        parts.append(f"Authors: {str(row['authors']).strip()}")

    if str(row['venue']).strip():
        parts.append(f"Venue: {str(row['venue']).strip()}")

    # Ghép các phần lại với nhau bằng dấu chấm
    return ". ".join(parts)

# Cập nhật cách áp dụng hàm này cho DataFrame
train_df['text_for_bert'] = train_df.apply(create_text_for_bert, axis=1)
test_df['text_for_bert'] = test_df.apply(create_text_for_bert, axis=1)

y = train_df['Label']

Đang tiền xử lý dữ liệu...


In [ ]:
# ==========================================
# 3. TRÍCH XUẤT ĐẶC TRƯNG NGỮ NGHĨA BẰNG SBERT
# ==========================================
print("Đang tải mô hình ngôn ngữ Sentence-BERT (lần đầu sẽ mất chút thời gian tải xuống)...")
model_st = SentenceTransformer('all-MiniLM-L6-v2')

print("Đang mã hóa dữ liệu Train thành Vector Ngữ Nghĩa...")
X_train_encoded = model_st.encode(train_df['text_for_bert'].tolist(), show_progress_bar=True)

print("Đang mã hóa dữ liệu Test thành Vector Ngữ Nghĩa...")
X_test_encoded = model_st.encode(test_df['text_for_bert'].tolist(), show_progress_bar=True)

In [ ]:
# ==========================================
# 4. HUẤN LUYỆN & VALIDATION
# ==========================================
# Split validation
X_train_split, X_val_split, y_train_split, y_val_split = train_test_split(
    X_train_encoded, y, test_size=0.2, random_state=42, stratify=y
)

print("\nĐang huấn luyện mô hình phân loại trên Vector Ngữ Nghĩa...")
# Với Dense Vectors của Neural Network, Logistic Regression lại hoạt động rất ổn định
classifier = LogisticRegression(class_weight='balanced', C=1.0, max_iter=1000, random_state=42)
classifier.fit(X_train_split, y_train_split)

# Đánh giá validation
y_val_pred = classifier.predict(X_val_split)
macro_f1 = f1_score(y_val_split, y_val_pred, average='macro')

print(f"\n✅ Validation Macro F1-Score (SBERT): {macro_f1:.4f}")
print("\nBáo cáo phân loại chi tiết:")
print(classification_report(y_val_split, y_val_pred))

Đang thiết lập Pipeline...


In [ ]:
# ==========================================
# 5. DỰ ĐOÁN TOÀN BỘ & TẠO FILE NỘP
# ==========================================
print("\nHuấn luyện lại trên toàn bộ tập train 500 mẫu...")
classifier.fit(X_train_encoded, y)

print("Đang dự đoán tập test...")
test_predictions = classifier.predict(X_test_encoded)

submission = pd.DataFrame({
    'id': test_df['id'],
    'Label': test_predictions
})

submission.to_csv('submission_sbert.csv', index=False)
print("🎉 Đã lưu kết quả ra file 'submission_sbert.csv'!")

Đang huấn luyện mô hình LinearSVC...

✅ Validation Macro F1-Score: 0.2830

Báo cáo phân loại chi tiết (Classification Report):
              precision    recall  f1-score   support

           1       0.39      0.35      0.37        26
           2       0.26      0.29      0.27        21
           3       0.06      0.06      0.06        17
           4       0.27      0.17      0.21        18
           5       0.44      0.60      0.51        20

    accuracy                           0.30       102
   macro avg       0.28      0.29      0.28       102
weighted avg       0.30      0.30      0.30       102



In [7]:
# ==========================================
# 6. HUẤN LUYỆN LẠI TRÊN TOÀN BỘ TẬP TRAIN VÀ DỰ ĐOÁN TEST SET
# ==========================================
print("\nHuấn luyện mô hình trên toàn bộ tập train để dự đoán test set...")
# Fit lại trên toàn bộ dữ liệu X, y để tận dụng tối đa 500 mẫu
model_pipeline.fit(X, y)

X_test = test_df[feature_cols]
test_predictions = model_pipeline.predict(X_test)


Huấn luyện mô hình trên toàn bộ tập train để dự đoán test set...


In [8]:
# ==========================================
# 7. TẠO FILE SUBMISSION
# ==========================================
submission = pd.DataFrame({
    'id': test_df['id'],
    'Label': test_predictions
})

# Lưu ra file CSV
submission.to_csv('my_submission.csv', index=False)
print("🎉 Đã lưu kết quả dự đoán ra file 'my_submission.csv'. Bạn có thể nộp file này lên hệ thống!")

🎉 Đã lưu kết quả dự đoán ra file 'my_submission.csv'. Bạn có thể nộp file này lên hệ thống!
